# HW0. Your first TPU run

This notebook has **no TPU attached**. That is deliberate.

Two people cannot share a Cloud TPU v5e chip. GKE schedules one pod per TPU node, and
that pod uses every chip on the node. If each of the 300 people in this course held a
chip for a full assignment, the course would need 300 chips. Most of that time nobody
would run code.

This notebook therefore runs on CPU. You write a script, and send it to a chip when it
is ready, the way `sbatch` works on a cluster. `submit_tpu.submit()` packs the folder
your script is in, puts it in a queue, and a free v5e chip runs it. `watch()` shows its
output as it runs. You hold the chip only while your script runs.

The first run of the day takes two to three minutes. The cluster scales to zero, so GKE
must build a machine for you. Later runs start faster, because a node from an earlier
job is usually still available.

Everything you submit goes in the `hw0/` folder. `submit()` sends the whole folder the
script is in, so keep scripts out of your home folder itself.

In [ ]:
import os
import submit_tpu

os.makedirs("hw0", exist_ok=True)

In [ ]:
%%writefile hw0/hello.py
import jax
print("devices:", jax.devices())
print("platform:", jax.devices()[0].platform)

In [ ]:
job = submit_tpu.submit("hw0/hello.py", dataset=None, follow=True)

You should see one `TpuDevice`. If you see a CPU device instead, something is wrong.
Tell a TA before you continue. Every number you measure after that point is a CPU
number with a TPU label on it.

Stopping the cell above only stops *watching*. The job keeps its place in line and
keeps running. `submit_tpu.status()` lists your jobs, and `submit_tpu.cancel()` stops
the active one.

## Part 1. Attention, and where the time goes

Below is a single multi-head attention forward pass, the shape that dominates an LLM
forward pass. Run both cells, then answer the questions underneath.

In [ ]:
%%writefile hw0/attention.py
import sys
import time
import jax, jax.numpy as jnp

S = int(sys.argv[1]) if len(sys.argv) > 1 else 1024
B, H, D = 8, 12, 64
key = jax.random.PRNGKey(0)
q, k, v = jax.random.normal(key, (3, B, H, S, D), dtype=jnp.bfloat16)

@jax.jit
def attn(q, k, v):
    logits = jnp.einsum("bhqd,bhkd->bhqk", q, k) / jnp.sqrt(D).astype(q.dtype)
    return jnp.einsum("bhqk,bhkd->bhqd", jax.nn.softmax(logits, axis=-1), v)

attn(q, k, v).block_until_ready()          # compile first, time second
t0 = time.perf_counter()
for _ in range(50):
    out = attn(q, k, v)
out.block_until_ready()
dt = (time.perf_counter() - t0) / 50

flops = 4 * B * H * S * S * D
print(f"device: {jax.devices()[0]}")
print(f"S={S}: {dt*1e3:.3f} ms per pass")
print(f"{flops/dt/1e12:.2f} TFLOP/s")

In [ ]:
job = submit_tpu.submit("hw0/attention.py", args=["1024"], dataset=None, follow=True)

**Q1.** A v5e chip peaks near 197 TFLOP/s in bfloat16. You measured far less than that.
Work out how many bytes the `logits` array occupies and how many bytes the chip must
move to compute the softmax over it. Is this kernel limited by arithmetic or by memory
bandwidth? Show the arithmetic intensity you calculated.

**Q2.** Run it again with `args=["2048"]`. Predict the change in time *before* you run
it, then explain any gap between your prediction and the measurement.

## Part 2. Make it faster

The two-einsum version above materialises the whole `B × H × S × S` logits array in
memory. Flash attention avoids that by tiling over the sequence and keeping a running
softmax, so the large intermediate never exists.

Use `jax.nn.dot_product_attention` and compare. Report both numbers and explain the
difference in terms of the bytes moved, not just the wall time.

In [ ]:
%%writefile hw0/flash.py
# Your code here. Write a complete script: this file is what runs on the chip.


In [ ]:
job = submit_tpu.submit("hw0/flash.py", dataset=None, follow=True)

## Checking your place in the queue

If a run is slow, it is either waiting for a chip or waiting for a machine.
`watch()` prints which: `queued, #N in line` means you are waiting behind N-1 other
jobs, and `starting` means a chip is reserved for you and a machine is being built.
That is normal and takes a couple of minutes. At any time:

In [ ]:
submit_tpu.status()

## Submitting

Hand in this notebook with all outputs kept. Every job prints the device it ran on, so
we can see the work happened on a real chip. `submit_tpu.logs(job)` returns a finished
job's full output if you need it again.